# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided as a Croissant schema via the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed.
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
meta = dataset.metadata
print("Name:", meta.name)
print("Identifier:", getattr(meta, 'identifier', ''))
print("Description:", meta.description)
print("License:", getattr(meta, 'license', ''))

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List all available record sets and their fields by @id
print("--- Available Record Sets ---")
for rs in dataset.metadata.record_sets:
    print(f"Record Set '@id': {rs.id}")
    print(f"  Name: {getattr(rs,'name', '')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field '@id': {field.id}, name: {getattr(field,'name','')}, data_type: {getattr(field,'data_type','')}")
    print("")

## 3. Data Extraction
Load records from a chosen record set (identified by its `@id`) into a pandas DataFrame for further exploration.

We'll use the first available record set for demonstration. You can change the selection as needed.

In [ ]:
# Prepare to extract all record sets into separate DataFrames
# Use record set @id for reference

record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_sets:
    main_record_set_id = record_sets[0]  # For demonstration, use the first one
    print(f"\nColumns in record set '@id' {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Let's apply data processing steps on the main record set, such as filtering records, normalizing numeric fields, and grouping by key attributes.

**Note:** Please update the selected field `@id`s as needed, based on the column list printed above.

In [ ]:
# Example EDA: Filtering and Normalization
# Please adapt 'numeric_field_id' and 'group_field_id' to your data

numeric_field_id = None
group_field_id = None

# Auto-select a numeric field if present
import numpy as np
if record_sets:
    df = dataframes[main_record_set_id]
    # Try to auto-detect numeric fields
    candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if candidates:
        numeric_field_id = candidates[0]
    else:
        # Try to convert columns to numeric if possible
        for col in df.columns:
            try:
                df_num = pd.to_numeric(df[col], errors='coerce')
                if df_num.notnull().sum() > 0:
                    df[col] = df_num
                    numeric_field_id = col
                    break
            except:
                continue

    # Select a group field if available
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < max(15, len(df)//10):
            group_field_id = col
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field detected for EDA. Please inspect your columns and edit the field variables above.")
else:
    print("No record sets in this dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset (e.g., histogram, boxplot, or scatter).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if record_sets and numeric_field_id:
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    sns.histplot(df[numeric_field_id].dropna(), ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field_id}")
    
    if group_field_id:
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, ax=axes[1])
        axes[1].set_title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
    else:
        df.boxplot(column=numeric_field_id, ax=axes[1])
        axes[1].set_title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot: No record set or numeric field available.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells using the `mlcroissant` library.

- We previewed available record sets and fields (referencing `@id`s throughout).
- We extracted the main record set into a DataFrame, and performed basic filtering, normalization, and grouping.
- We visualized the distribution of a numeric field and its comparison across a group.

You can further repeat this process for other record sets or fields of interest. For best results with domain-specific analysis, review field descriptions and units as provided in the dataset metadata.